In [13]:
import pandas as pd
import joblib
df = pd.read_csv('../datasets/tas2_yontem1.csv')


In [14]:
# import pandas as pd
# import numpy as np
# from datetime import timedelta

# # Simüle edilmiş veri: 2 formID, her biri 10 zaman noktasında
# np.random.seed(42)

# def generate_data(form_id, base_price, trend, noise=5):
#     dates = pd.date_range(start="2024-01-01", periods=10, freq='D')
#     trend_line = base_price + trend * np.arange(10) + np.random.normal(0, noise, 10)
#     df = pd.DataFrame({
#         'formID': form_id,
#         'createdAt': dates,
#         'min_price': trend_line - 10,
#         'max_price': trend_line + 10
#     })
#     return df

# df1 = generate_data('A', 100, 2)   # Artan trend
# df2 = generate_data('B', 200, -1)  # Azalan trend
# df = pd.concat([df1, df2], ignore_index=True)


# from sklearn.linear_model import LinearRegression

# df_trend = df.copy()
# df_trend['createdAt'] = pd.to_datetime(df_trend['createdAt'])
# df_trend = df_trend.sort_values('createdAt')

# # Zamanı normalize ediyoruz (gün cinsinden fark)
# df_trend['timestamp'] = (df_trend['createdAt'] - df_trend['createdAt'].min()).dt.total_seconds() / (60 * 60 * 24)

# # Ek sütunlar
# df_trend['avg_price'] = df_trend[['min_price', 'max_price']].mean(axis=1)
# df_trend['price_range'] = df_trend['max_price'] - df_trend['min_price']

# def detect_trend(group, threshold=0.01):
#     result = {}
#     for col in ['min_price', 'max_price', 'avg_price', 'price_range']:
#         if len(group) < 2:
#             result[f'{col}_slope'] = 0
#             result[f'{col}_r2'] = 0
#             result[f'{col}_trend'] = 'durağan'
#             continue

#         X = group['timestamp'].values.reshape(-1, 1)
#         y = group[col].values

#         model = LinearRegression().fit(X, y)
#         slope = model.coef_[0]
#         r2 = model.score(X, y)

#         if slope > threshold:
#             trend = 'artan'
#         elif slope < -threshold:
#             trend = 'azalan'
#         else:
#             trend = 'durağan'

#         result[f'{col}_slope'] = slope
#         result[f'{col}_r2'] = r2
#         result[f'{col}_trend'] = trend

#     return pd.Series(result)

# trend_summary = df_trend.groupby('formID').apply(detect_trend)
# print(trend_summary)

# import matplotlib.pyplot as plt

# def plot_form_trend(form_id):
#     subset = df_trend[df_trend['formID'] == form_id]
#     plt.figure(figsize=(10, 4))
#     plt.plot(subset['createdAt'], subset['avg_price'], label='Avg Price')
#     plt.title(f"Form ID: {form_id} - Avg Price Over Time")
#     plt.xlabel("Tarih")
#     plt.ylabel("Fiyat")
#     plt.grid(True)
#     plt.legend()
#     plt.show()

# plot_form_trend('A')
# plot_form_trend('B')


In [15]:
df.head()
df.drop(columns=["records","pageID","date", "formID",'createdAt', 'updatedAt', 'createdAt_session', 'updatedAt_session',"sessionStart"], inplace=True, errors="ignore")
yontem1df = df.copy()
yontem1df['max_price'] = df['max_price'].fillna(df['max_price'].median())
yontem1df['min_price'] = df['min_price'].fillna(df['min_price'].median())
yontem1df['gender'] = df['gender'].fillna("")
yontem1df['message'] = df['message'].fillna("")
yontem1df['comment'] = df['comment'].fillna("")
yontem1df['deviceType'] = df['deviceType'].fillna("")
yontem1df['categories'] = df['categories'].fillna("")
yontem1df['contents'] = df['contents'].fillna("")
df = yontem1df.copy()


In [16]:

from sklearn.linear_model import LinearRegression

df_trend = df.copy()
df_trend["createdAt_form"] =     dates = pd.date_range(start="2024-01-01", periods=110, freq='D')

df_trend['createdAt_form'] = pd.to_datetime(df_trend['createdAt_form'])
df_trend = df_trend.sort_values('createdAt_form')

# Zamanı normalize ediyoruz (gün cinsinden fark)
df_trend['timestamp'] = (df_trend['createdAt_form'] - df_trend['createdAt_form'].min()).dt.total_seconds() / (60 * 60 * 24)

# Ek sütunlar
df_trend['avg_price'] = df_trend[['min_price', 'max_price']].mean(axis=1)
df_trend['price_range'] = df_trend['max_price'] - df_trend['min_price']

def detect_trend(group, threshold=0.01):
    result = {}
    for col in ['min_price', 'max_price', 'avg_price', 'price_range']:
        if len(group) < 2:
            result[f'{col}_slope'] = 0
            result[f'{col}_r2'] = 0
            result[f'{col}_trend'] = 'durağan'
            continue

        X = group['timestamp'].values.reshape(-1, 1)
        y = group[col].values

        model = LinearRegression().fit(X, y)
        slope = model.coef_[0]
   

        if slope > threshold:
            trend = 'artan'
        elif slope < -threshold:
            trend = 'azalan'
        else:
            trend = 'durağan'

        result[f'{col}_slope'] = slope
    
        result[f'{col}_trend'] = trend

    return pd.Series(result)

trend_summary = df_trend.groupby('sessionID').apply(detect_trend)

trend_summary
 

/tmp/ipykernel_184381/1166959878.py:45: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  trend_summary = df_trend.groupby('sessionID').apply(detect_trend)


sessionID                                              
2c8b8f97-c31c-466f-86c1-03c98760b78a  min_price_slope          0.0
                                      min_price_trend      durağan
                                      max_price_slope          0.0
                                      max_price_trend      durağan
                                      avg_price_slope          0.0
                                                            ...   
dab35cbb-1d74-4220-b694-ad448dfcc806  avg_price_r2               0
                                      avg_price_trend      durağan
                                      price_range_slope          0
                                      price_range_r2             0
                                      price_range_trend    durağan
Length: 76, dtype: object

In [ ]:
import pandas as pd
import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import FunctionTransformer
from sklearn.compose import ColumnTransformer

def prepare_trend_summary(df):

    def prepare_time_features(df):
       
        df['createdAt_form'] = pd.to_datetime(df['createdAt_form'])
        df = df.sort_values('createdAt_form')

        df['timestamp'] = (df['createdAt_form'] - df['createdAt_form'].min()).dt.total_seconds() / (60 * 60 * 24)
        df['avg_price'] = df[['min_price', 'max_price']].mean(axis=1)
        df['price_range'] = df['max_price'] - df['min_price']
        return df

    prepare_fn = FunctionTransformer(prepare_time_features)


    class TrendExtractor(BaseEstimator, TransformerMixin):
        def __init__(self, value_column, time_column='timestamp', threshold=0.01):
            self.value_column = value_column
            self.time_column = time_column
            self.threshold = threshold

        def fit(self, X, y=None):
            return self

        def transform(self, X):
            results = []
            for session_id, group in X.groupby('sessionID'):
                group = group.sort_values(self.time_column)
                if len(group) < 2:
                    slope = 0
                    trend = 'durağan'
                else:
                    X_t = group[self.time_column].values.reshape(-1, 1)
                    y_t = group[self.value_column].values
                    model = LinearRegression().fit(X_t, y_t)
                    slope = model.coef_[0]
                    if slope > self.threshold:
                        trend = 'increasing'
                    elif slope < -self.threshold:
                        trend = 'decreasing'
                    else:
                        trend = 'stable'

                results.append({
                    'sessionID': session_id,
                    f'{self.value_column}_slope': slope,
                    f'{self.value_column}_trend': trend
                })

            return pd.DataFrame(results)



    min_trend = TrendExtractor('min_price')
    max_trend = TrendExtractor('max_price')
    avg_trend = TrendExtractor('avg_price')
    range_trend = TrendExtractor('price_range')

    df = df.copy()
    df['createdAt_form'] = pd.date_range("2024-01-01", periods=len(df), freq='D')



    df_ready = prepare_time_features(df)

    results = [
        min_trend.transform(df_ready),
        max_trend.transform(df_ready),
        avg_trend.transform(df_ready),
        range_trend.transform(df_ready)
    ]

    from functools import reduce
    trend_summary = reduce(lambda left, right: pd.merge(left, right, on='sessionID'), results)
    return trend_summary

In [35]:
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.cluster import KMeans

numerical_features = trend_summary.select_dtypes(include=['float64', 'int64']).columns
trend_summary = prepare_trend_summary(df)
X = trend_summary[numerical_features].copy()

kmeans = KMeans(n_clusters=3, random_state=42, n_init='auto')
ss = StandardScaler()
ohe = OneHotEncoder(handle_unknown='ignore')
preprocessor = ColumnTransformer(
    transformers=[
        ('num', ss, numerical_features)
    ]
)
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('clusterer', kmeans)
])
trend_summary['cluster'] = pipeline.fit_predict(X)


trend_summary['cluster'].value_counts().sort_index().to_dict()

/home/hyo/Documents/projects/trackerv2/.venv/lib/python3.13/site-packages/sklearn/base.py:1363: ConvergenceWarning: Number of distinct clusters (2) found smaller than n_clusters (3). Possibly due to duplicate points in X.
  return fit_method(estimator, *args, **kwargs)


{0: 8, 1: 1}